# Data Prep for SVM

In [56]:
import polars as pl
from run_config import (
    PATHS,
    RUN_MODE,
    MODEL_START_DATE,
    MODEL_END_DATE,
    H3_RESOLUTION
)
from datetime import datetime

## Train Test Split

Here we perform the split into train and test data, with a **split** of 70% train, 30% test data. The split is performed **random**. 

In [57]:
# falls was geändert werden muss schreiben, bitte schreiben 
ALL = True # option all, all possible combination are run
SPATIAL_UNIT = "community_area" # option: census_tract, community_area, h3_cell
TIME_UNIT = "4h" # options: 4h, 2h, 1h
RANDOM = True
MODE = "full" # options: sample, full
OUTPUT = "../data/" + MODE + "/train_test_data/"

In [58]:
if SPATIAL_UNIT not in {"census_tract", "community_area", "h3_cell"}:
    SPATIAL_UNIT = "community_area"
    print("No valid spatial unit was given, default: Community Area was used")

DEMAND_PATHS = {
    "1h": {
        "census_tract": PATHS.gold_1h_demand_census_tracts,
        "community_area": PATHS.gold_1h_demand_community_areas,
        "h3_cell": PATHS.gold_1h_demand_hexagon,
    },
    "2h": {
        "census_tract": PATHS.gold_2h_demand_census_tracts,
        "community_area": PATHS.gold_2h_demand_community_areas,
        "h3_cell": PATHS.gold_2h_demand_hexagon,
    },
    "4h": {
        "census_tract": PATHS.gold_4h_demand_census_tracts,
        "community_area": PATHS.gold_4h_demand_community_areas,
        "h3_cell": PATHS.gold_4h_demand_hexagon,
    },
}

SEED = 42

In [59]:
MODEL_START_TS = datetime.fromisoformat(MODEL_START_DATE)
MODEL_END_TS = datetime.fromisoformat(MODEL_END_DATE)

In [60]:
if ALL:
    time_units = list(DEMAND_PATHS.keys())
    spatial_units = list(next(iter(DEMAND_PATHS.values())).keys())

In [61]:

if ALL == True:
    for TU in time_units:     
        for SU in spatial_units:
            DATASET = DEMAND_PATHS[TU][SU]
            if(RANDOM == True):
                
                # split randomly
                df_split = (
                    pl.scan_parquet(DATASET)
                    .with_row_index("_row_id")
                    .with_columns(
                        (pl.col("_row_id").hash(seed=SEED) % 100).alias("_split_bucket")
                    )
                )

                df_split = df_split.filter(
                    (pl.col("datetime_hour") >= MODEL_START_TS)
                    & (pl.col("datetime_hour") < MODEL_END_TS)
                )
                
                train = (
                    df_split
                    .filter(pl.col("_split_bucket") < 70)
                    .drop(["_row_id", "_split_bucket"])
                )

                test = (
                    df_split
                    .filter(pl.col("_split_bucket") >= 70)
                    .drop(["_row_id", "_split_bucket"])
                )
            else :
                # split according to time
                df_split = pl.scan_parquet(DATASET)

                train = df_split.filter(
                    pl.col("datetime_hour") < pl.datetime(2025, 9, 1)
                )

                test = df_split.filter(
                    pl.col("datetime_hour") >= pl.datetime(2025, 9, 1)
                )


            total_count = df_split.select(pl.len()).collect().item()
            train_count = train.select(pl.len()).collect().item()
            test_count = test.select(pl.len()).collect().item()

            print("Total:", total_count)
            print("Train:", train_count, " Share: ", round(train_count / total_count,2))
            print("Test:", test_count, " Share: ", round(test_count / total_count, 2))
            
            print("Created parquets for Time Unit:" + TU + " and Spatial Unit:" + SU)

            if SU == "h3_cell":
                train.sink_parquet(f"{OUTPUT}svm_{SU}_{str(H3_RESOLUTION)}_{TU}_train.parquet")
                test.sink_parquet(f"{OUTPUT}svm_{SU}_{str(H3_RESOLUTION)}_{TU}_test.parquet")
            else: 
                train.sink_parquet(OUTPUT + "svm_" + SU + "_" + TU + "_train.parquet")
                test.sink_parquet(OUTPUT + "svm_" + SU + "_" + TU + "_test.parquet")
else:
    DATASET = DEMAND_PATHS[TIME_UNIT][SPATIAL_UNIT] 
    if(RANDOM == True):
        # split randomly
        df_split = (
            pl.scan_parquet(DATASET)
            .with_row_index("_row_id")
            .with_columns(
                (pl.col("_row_id").hash(seed=SEED) % 100).alias("_split_bucket")
            )
        )

        train = (
            df_split
            .filter(pl.col("_split_bucket") < 70)
            .drop(["_row_id", "_split_bucket"])
        )

        test = (
            df_split
            .filter(pl.col("_split_bucket") >= 70)
            .drop(["_row_id", "_split_bucket"])
        )
    else :
        # split according to time
        df_split = pl.scan_parquet(DATASET)

        train = df_split.filter(
            pl.col("datetime_hour") < pl.datetime(2025, 9, 1)
        )

        test = df_split.filter(
            pl.col("datetime_hour") >= pl.datetime(2025, 9, 1)
        )


    total_count = df_split.select(pl.len()).collect().item()
    train_count = train.select(pl.len()).collect().item()
    test_count = test.select(pl.len()).collect().item()

    print("Total:", total_count)
    print("Train:", train_count, " Share: ", round(train_count / total_count,2))
    print("Test:", test_count, " Share: ", round(test_count / total_count, 2))
    
    if SPATIAL_UNIT == "h3_cell":
        train.sink_parquet(f"{OUTPUT}svm_{SPATIAL_UNIT}_{str(H3_RESOLUTION)}_{TIME_UNIT}_test.parquet")
        test.sink_parquet(f"{OUTPUT}svm_{SPATIAL_UNIT}_{str(H3_RESOLUTION)}_{TIME_UNIT}_test.parquet")
    else: 
        train.sink_parquet(OUTPUT + "svm_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_train.parquet")
        test.sink_parquet(OUTPUT + "svm_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_test.parquet")

Total: 10219920
Train: 7153261  Share:  0.7
Test: 3066659  Share:  0.3
Created parquets for Time Unit:1h and Spatial Unit:census_tract


Total: 896280
Train: 626697  Share:  0.7
Test: 269583  Share:  0.3
Created parquets for Time Unit:1h and Spatial Unit:community_area
Total: 9928920
Train: 6952269  Share:  0.7
Test: 2976651  Share:  0.3
Created parquets for Time Unit:1h and Spatial Unit:h3_cell
Total: 5109960
Train: 3576280  Share:  0.7
Test: 1533680  Share:  0.3
Created parquets for Time Unit:2h and Spatial Unit:census_tract
Total: 448140
Train: 313428  Share:  0.7
Test: 134712  Share:  0.3
Created parquets for Time Unit:2h and Spatial Unit:community_area
Total: 4964460
Train: 3474009  Share:  0.7
Test: 1490451  Share:  0.3
Created parquets for Time Unit:2h and Spatial Unit:h3_cell
Total: 2554980
Train: 1788655  Share:  0.7
Test: 766325  Share:  0.3
Created parquets for Time Unit:4h and Spatial Unit:census_tract
Total: 224070
Train: 156865  Share:  0.7
Test: 67205  Share:  0.3
Created parquets for Time Unit:4h and Spatial Unit:community_area
Total: 2482230
Train: 1736205  Share:  0.7
Test: 746025  Share:  0.3
Created 

In [62]:
df_split.head(10).collect()

_row_id,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,tmpc,relh,sknt,vsby,p01m,skyc1_BKN,skyc1_CLR,skyc1_FEW,skyc1_OVC,skyc1_SCT,skyc1_VV,date,is_holiday,h3_cell,food_drink,landmark,shop,train_station,trip_count,trip_seconds_sum,trip_seconds_mean,trip_seconds_min,trip_seconds_max,trip_miles_sum,trip_miles_mean,trip_miles_min,trip_miles_max,fare_sum,fare_mean,fare_min,fare_max,tips_sum,tips_mean,tips_min,tips_max,tolls_sum,tolls_mean,tolls_min,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type,_split_bucket
u32,datetime[μs],i8,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,i8,i8,i8,date,i8,str,f64,f64,f64,f64,u32,i64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,u64
133,2025-01-07 04:00:00,1,2,4,0.0,1.0,0.781831,0.62349,0.866025,0.5,-3.998,61.402,11.4,10.0,0.0,0,0,0,1,0,0,2025-01-07,0,"""882664d997fffff""",5.0,1.0,4.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",54
135,2025-01-02 04:00:00,1,4,4,0.0,1.0,0.433884,-0.900969,0.866025,0.5,-3.668,59.362,11.8,9.6,0.0003,0,0,0,1,0,0,2025-01-02,0,"""882664d997fffff""",5.0,1.0,4.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",26
138,2025-01-06 16:00:00,1,1,16,0.0,1.0,0.0,1.0,-0.866025,-0.5,-0.28,72.0325,16.0,9.25,0.0004,1,0,0,0,0,0,2025-01-06,0,"""882664d997fffff""",5.0,1.0,4.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",11
170,2025-03-10 04:00:00,3,1,4,0.866025,0.5,0.0,1.0,0.866025,0.5,8.703333,48.813333,7.666667,10.0,0.0,0,0,1,0,0,0,2025-03-10,0,"""882664d997fffff""",5.0,1.0,4.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",25
171,2025-04-02 12:00:00,4,3,12,1.0,6.1232e-17,0.974928,-0.222521,1.2246e-16,-1.0,6.945,86.432,12.4,2.8,16.0,1,0,0,0,0,0,2025-04-02,0,"""882664d997fffff""",5.0,1.0,4.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",5
172,2025-04-16 08:00:00,4,3,8,1.0,6.1232e-17,0.974928,-0.222521,0.866025,-0.5,3.11,66.892,4.8,10.0,0.0,0,0,1,0,0,0,2025-04-16,0,"""882664d997fffff""",5.0,1.0,4.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",16
173,2025-04-24 04:00:00,4,4,4,1.0,6.1232e-17,0.433884,-0.900969,0.866025,0.5,14.445,68.66,2.25,10.0,0.0,1,0,0,0,0,0,2025-04-24,0,"""882664d997fffff""",5.0,1.0,4.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",80
174,2025-03-06 16:00:00,3,4,16,0.866025,0.5,0.433884,-0.900969,-0.866025,-0.5,3.7475,38.8575,11.75,10.0,0.0,0,1,0,0,0,0,2025-03-06,0,"""882664d997fffff""",5.0,1.0,4.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",6
175,2025-04-13 16:00:00,4,7,16,1.0,6.1232e-17,-0.781831,0.62349,-0.866025,-0.5,16.67,33.5725,15.5,10.0,0.0,0,0,1,0,0,0,2025-04-13,0,"""882664d997fffff""",5.0,1.0,4.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",90
